Step 1: Starter Code

In [ ]:
"""
Product Description Generator - STARTER CODE (Needs Refactoring)
This code works but has many issues that need to be fixed.
"""
 
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field, validator
from typing import List, Optional
 
class Product(BaseModel):
    id: str
    name: str
    category: str
    price: float
    features: List[str] = []
    
    @validator('price')
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError('Price must be positive')
        return v
 
def generate_product_descriptions(json_file):
    # Load JSON file
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Validate products
    products = []
    for item in data.get('products', []):
        try:
            product = Product(**item)
            products.append(product)
        except:
            pass  # Silent failure!
    
    # Generate descriptions
    client = OpenAI(api_key="your-api-key-here")
    results = []
    
    for product in products:
        # Create prompt
        prompt = f"""Create a product description for:
Name: {product.name}
Category: {product.category}
Price: ${product.price}
Features: {', '.join(product.features)}
 
Generate a compelling product description."""
        
        # Call API
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Process response
        description = response.choices[0].message.content
        results.append({
            "product_id": product.id,
            "name": product.name,
            "description": description
        })
    
    # Save results
    with open('results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    return results
 
# Usage
if __name__ == "__main__":
    generate_product_descriptions("products.json")

## Original Code Overview

The starter code is a single-function script that loads a JSON file of products, validates each one with a Pydantic model, calls the OpenAI API to generate a description for each, and saves the results to `results.json`. All of this happens inside one function: `generate_product_descriptions()`.

## Where It Fails Silently

The most critical silent failure is the bare `except: pass` in the validation loop — if a product has a missing field, wrong type, or negative price, it is dropped with no message, no count, and no trace. The file-loading block has no error handling at all: a missing `products.json` throws a raw `FileNotFoundError` traceback with no context, and malformed JSON throws `json.JSONDecodeError` with no hint of where the syntax error is. The API call has no try/except, so a rate limit, timeout, or bad API key crashes the entire loop mid-run and any results already generated are lost because saving only happens at the very end. Finally, `response.choices[0]` is accessed without checking whether `choices` is empty, which can happen on content filter triggers and throws an `IndexError` with no explanation.

## Concerns Mixed Together in One Function

`generate_product_descriptions()` does six things that should be separate: file loading, JSON parsing, Pydantic validation, prompt construction, API calling, and result saving. This means none of these can be tested independently — you cannot test prompt formatting without triggering a real API call, and you cannot test validation without loading a file first.

## Error Handling Gaps

| Location | Error type | Status |
|---|---|---|
| `open(json_file)` | `FileNotFoundError` | ❌ unhandled |
| `json.load(f)` | `json.JSONDecodeError` | ❌ unhandled |
| `Product(**item)` | `pydantic.ValidationError` | ❌ silently swallowed |
| `OpenAI(api_key=...)` | `AuthenticationError` | ❌ unhandled |
| `client.chat.completions.create()` | `APIError`, `RateLimitError`, `Timeout` | ❌ unhandled |
| `response.choices[0]` | `IndexError` | ❌ unhandled |
| `open('results.json', 'w')` | `PermissionError`, `OSError` | ❌ unhandled |

# Starter Code – Issues to Fix

## Analysis Questions

**1. What happens if `products.json` does not exist?**
Python throws a raw `FileNotFoundError` traceback and the entire script crashes immediately. There is no message about which file was expected, no hint of the current working directory, and no suggestion of what to check.

**2. What happens if JSON is invalid?**
`json.load(f)` throws `json.JSONDecodeError` with a low-level message that includes a line number but no context about which file failed or what to do about it. The script crashes before any products are processed.

**3. What happens if product validation fails?**
Nothing visible. The bare `except: pass` swallows the `ValidationError` completely. The invalid product is silently dropped from the list. If every product fails validation, the script continues happily with an empty list, makes zero API calls, and saves an empty `results.json` — with no indication that anything went wrong.

**4. What happens if the API call fails?**
The script crashes with a raw OpenAI traceback. There is no retry logic, no message identifying which product caused the failure, and no partial save — any descriptions already generated in the loop are lost because `results.json` is only written at the very end.

**5. What functions do multiple things?**
`generate_product_descriptions()` does all of the following in one block:
- Opens and reads a file
- Parses JSON
- Validates each product with Pydantic
- Creates the OpenAI client
- Builds a prompt string
- Makes an API call
- Parses the API response
- Formats the output dict
- Saves results to disk

None of these can be tested in isolation.

**6. Where is code repeated that could be a helper function?**
- The prompt string is built inline inside the loop — it should be a
  `create_product_prompt(product)` helper so the template lives in one place and can be tested or changed independently
- The response parsing `response.choices[0].message.content` is inline —
  it should be a `parse_api_response(response)` helper so the `IndexError` risk is handled once, not wherever the API is called
- The output dict `{"product_id": ..., "name": ..., "description": ...}` is assembled inline — it should be a `format_output(product, description)` helper

## Summary: Issues to Fix

| 1 | No error handling on file open | `open(json_file)` |
| 2 | No error handling on JSON parse | `json.load(f)` |
| 3 | Silent failure swallows all validation errors | `except: pass` |
| 4 | No error handling on API call | `client.chat.completions.create()`
| 5 | No partial save if API loop crashes mid-run | end of function |
| 6 | `response.choices[0]` unguarded — crashes on empty choices | response parsing |
| 7 | API key hardcoded as literal string | `OpenAI(api_key=...)` |
| 8 | One function does nine separate things | `generate_product_descriptions()` |
| 9 | Prompt template repeated inline in loop | inside `for product in products` |
| 10 | Response parsing repeated inline in loop | inside `for product in products` |
| 11 | Output formatting repeated inline in loop | inside `for product in products` |
| 12 | `@validator` is Pydantic v1 syntax, broken in v2 | `Product` model |

In [1]:
# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

import json
import os
from typing import Optional
from pydantic import BaseModel, field_validator, ValidationError
from typing import List


# ── Pydantic Model ─────────────────────────────────────────────────────────────

class Product(BaseModel):
    id: str
    name: str
    category: str
    price: float
    features: List[str] = []

    @field_validator('price')                      # Pydantic v2 syntax
    @classmethod
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError('Price must be positive')
        return v


# ── Helper 1: Load JSON ─────────────────────────────────────────────────────────

def load_json_file(file_path: str) -> dict:
    """Load and parse JSON file with error handling."""
    try:
        with open(file_path, 'r') as f:
            return json.load(f)

    except FileNotFoundError:
        print(
            f"ERROR in load_json_file(): FileNotFoundError\n"
            f"  Location: File '{file_path}' not found\n"
            f"  Current directory: {os.getcwd()}\n"
            f"  Suggestion: Check the file path and confirm the file exists"
        )
        raise

    except json.JSONDecodeError as e:
        print(
            f"ERROR in load_json_file(): JSONDecodeError\n"
            f"  Location: File '{file_path}', line {e.lineno}, column {e.colno}\n"
            f"  Message: {e.msg}\n"
            f"  Suggestion: Fix the JSON syntax at the line shown above"
        )
        raise


# ── Helper 2: Validate Product ─────────────────────────────────────────────────

def validate_product_data(product_dict: dict) -> Optional[Product]:
    """Validate product data using Pydantic. Returns Product or None on failure."""
    try:
        return Product(**product_dict)

    except ValidationError as e:
        error_details = "\n".join(
            f"    - field '{'.'.join(str(loc) for loc in err['loc'])}': {err['msg']}"
            for err in e.errors()
        )
        print(
            f"ERROR in validate_product_data(): ValidationError\n"
            f"  Product ID: {product_dict.get('id', 'unknown')}\n"
            f"  Invalid fields:\n{error_details}\n"
            f"  Suggestion: Fix the fields listed above in your JSON data"
        )
        return None

    except Exception as e:
        print(
            f"ERROR in validate_product_data(): Unexpected error\n"
            f"  Product: {product_dict}\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check that product_dict is a plain dictionary"
        )
        return None


# ── Helper 3: Create Prompt ────────────────────────────────────────────────────

def create_product_prompt(product: Product) -> str:
    """Build the OpenAI prompt string for a product."""
    return (
        f"Create a compelling product description for an online store listing.\n\n"
        f"Product name: {product.name}\n"
        f"Category: {product.category}\n"
        f"Price: ${product.price:.2f}\n"
        f"Key features: {', '.join(product.features)}\n\n"
        f"Write 2-3 sentences. Focus on benefits, not just features. "
        f"Tone: enthusiastic but professional."
    )


# ── Helper 4: Parse API Response ───────────────────────────────────────────────

def parse_api_response(response) -> str:
    """Extract text content from an OpenAI API response object."""
    try:
        if not response.choices:
            print(
                f"ERROR in parse_api_response(): Empty choices list\n"
                f"  Location: response.choices is empty\n"
                f"  Message: The API returned no completion\n"
                f"  Suggestion: This can happen on content filter triggers — "
                f"check your prompt for policy violations"
            )
            return ""

        content = response.choices[0].message.content
        if content is None:
            print(
                f"ERROR in parse_api_response(): Content is None\n"
                f"  Location: response.choices[0].message.content\n"
                f"  Suggestion: The model returned a null response — try again"
            )
            return ""

        return content.strip()

    except AttributeError as e:
        print(
            f"ERROR in parse_api_response(): AttributeError\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check that you passed a valid OpenAI response object"
        )
        return ""


# ── Helper 5: Format Output ────────────────────────────────────────────────────

def format_output(product: Product, description: str) -> dict:
    """Format a product and its generated description into the output dict."""
    return {
        "product_id": product.id,
        "name": product.name,
        "category": product.category,
        "price": product.price,
        "description": description
    }

In [3]:
# ── Create sample data files ───────────────────────────────────────────────────

import json

# Valid products
products_data = {
    "products": [
        {
            "id": "P001",
            "name": "Wireless Bluetooth Headphones",
            "category": "Electronics",
            "price": 99.99,
            "features": ["Bluetooth 5.0", "Noise Cancelling", "30hr Battery"]
        },
        {
            "id": "P002",
            "name": "Smart Watch",
            "category": "Wearables",
            "price": 249.99,
            "features": ["Heart Rate Monitor", "GPS", "Water Resistant"]
        },
        {
            "id": "P003",
            "name": "Laptop Stand",
            "category": "Accessories",
            "price": 49.99,
            "features": ["Adjustable Height", "Aluminum", "Cable Management"]
        }
    ]
}

with open("products.json", "w") as f:
    json.dump(products_data, f, indent=2)
print("✅ products.json created")

# Invalid products (for testing error handling)
invalid_data = {
    "products": [
        {
            "id": "P004",
            "name": "Broken Item",
            "category": "Electronics",
            "price": -10.00        # negative price — should trigger validator
        },
        {
            "id": "P005",
            "category": "Electronics",
            "price": 50.00         # missing required field: name
        }
    ]
}

with open("invalid_products.json", "w") as f:
    json.dump(invalid_data, f, indent=2)
print("✅ invalid_products.json created")

# Malformed JSON (for testing JSONDecodeError)
with open("malformed.json", "w") as f:
    f.write('{"products": [}')    # intentionally broken syntax
print("✅ malformed.json created")

✅ products.json created
✅ invalid_products.json created
✅ malformed.json created


In [4]:
# ── TESTS ─────────────────────────────────────────────────────────
# Run each cell below separately to verify each helper before wiring them together

# ── Test load_json_file ────────────────────────────────────────────────────────

# Should succeed
data = load_json_file("products.json")
print("✅ Loaded:", data)

# Should print clear FileNotFoundError
try:
    load_json_file("missing.json")
except FileNotFoundError:
    print("✅ FileNotFoundError caught and described")

# Should print clear JSONDecodeError (create a file with bad JSON first)
# with open("malformed.json", "w") as f: f.write('{"products": [}')
# try:
#     load_json_file("malformed.json")
# except json.JSONDecodeError:
#     print("✅ JSONDecodeError caught and described")

✅ Loaded: {'products': [{'id': 'P001', 'name': 'Wireless Bluetooth Headphones', 'category': 'Electronics', 'price': 99.99, 'features': ['Bluetooth 5.0', 'Noise Cancelling', '30hr Battery']}, {'id': 'P002', 'name': 'Smart Watch', 'category': 'Wearables', 'price': 249.99, 'features': ['Heart Rate Monitor', 'GPS', 'Water Resistant']}, {'id': 'P003', 'name': 'Laptop Stand', 'category': 'Accessories', 'price': 49.99, 'features': ['Adjustable Height', 'Aluminum', 'Cable Management']}]}
ERROR in load_json_file(): FileNotFoundError
  Location: File 'missing.json' not found
  Current directory: c:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\Ironhack_Day7
  Suggestion: Check the file path and confirm the file exists
✅ FileNotFoundError caught and described


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()   # reads the .env file in the current folder

# confirm it loaded correctly
print("API key found:", os.getenv("OPENAI_API_KEY") is not None)

API key found: True


In [7]:
# ── STEP 3: MODULAR FUNCTIONS ─────────────────────────────────────────────────
#
# Each function below has exactly ONE job.
# The main() function at the bottom only orchestrates; it does no work itself.
# ─────────────────────────────────────────────────────────────────────────────

from openai import OpenAI, APIError
import os


# ── Module 1: Load + Validate ──────────────────────────────────────────────────
# Single responsibility: turn a file path into a clean list of Product objects.
# Combines load_json_file() and validate_product_data() from Step 2.
# Invalid products are skipped (with a printed error) rather than crashing the run.

def load_and_validate_products(json_path: str) -> List[Product]:
    """Load JSON file and return a list of valid Product objects."""

    # Step 1: load the raw file — load_json_file() handles FileNotFoundError
    # and JSONDecodeError and will raise if the file cannot be read
    raw_data = load_json_file(json_path)

    # Step 2: iterate over the products list inside the JSON
    # If the key 'products' is missing we get an empty list — no crash
    products = []
    for item in raw_data.get("products", []):

        # validate_product_data() returns a Product on success or None on failure
        # It also prints exactly which fields are invalid, so we don't need to here
        product = validate_product_data(item)

        if product is not None:
            products.append(product)

    # Report how many products survived validation so silent drops are visible
    print(f"ℹ️  Loaded {len(products)} valid product(s) from '{json_path}'")
    return products


# ── Module 2: Generate One Description ────────────────────────────────────────
# Single responsibility: call the API for ONE product and return the description.
# Combines create_product_prompt() and parse_api_response() from Step 2.
# Raises on API failure so the caller (process_products) decides how to handle it.

def generate_description(product: Product, api_client: OpenAI) -> str:
    """Call the OpenAI API and return a description string for one product."""

    # Build the prompt using the helper — template lives in one place only
    prompt = create_product_prompt(product)

    try:
        # Make the API call — this is the ONLY place an API call happens
        response = api_client.chat.completions.create(
            model="gpt-4o-mini",           # cheaper model; swap to gpt-4 if needed
            messages=[{"role": "user", "content": prompt}]
        )

        # Extract the text using the helper — IndexError risk is handled inside it
        return parse_api_response(response)

    except APIError as e:
        # Print WHERE the failure happened (function name + which product)
        # then re-raise so process_products() can decide to skip or abort
        print(
            f"ERROR in generate_description(): APIError\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Status code: {getattr(e, 'status_code', 'N/A')}\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check your API key, rate limits, or try again later"
        )
        raise


# ── Module 3: Process All Products ────────────────────────────────────────────
# Single responsibility: loop over every product and collect results.
# This is the ONLY function that knows about the full list.
# It calls generate_description() per product and format_output() per result.
# Products that fail the API call are skipped so one failure doesn't lose everything.

def process_products(products: List[Product], api_client: OpenAI) -> List[dict]:
    """Generate descriptions for all products and return a list of result dicts."""

    results = []

    for product in products:
        print(f"⏳ Generating description for '{product.name}'...")

        try:
            # generate_description() raises on API failure — caught here per product
            description = generate_description(product, api_client)

            # format_output() assembles the result dict — formatting lives in one place
            result = format_output(product, description)
            results.append(result)

            print(f"✅ Done: '{product.name}'")

        except APIError:
            # Error message already printed inside generate_description()
            # Skip this product and continue — partial results are still saved
            print(f"⚠️  Skipping '{product.name}' due to API error — continuing...\n")

    print(f"ℹ️  Processed {len(results)} of {len(products)} product(s) successfully")
    return results


# ── Module 4: Save Results ─────────────────────────────────────────────────────
# Single responsibility: write the results list to disk as JSON.
# Nothing else — no formatting, no API calls, no validation.

def save_results(results: List[dict], output_path: str) -> None:
    """Write results list to a JSON file at output_path."""

    try:
        with open(output_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"✅ Results saved to '{output_path}' ({len(results)} item(s))")

    except PermissionError:
        print(
            f"ERROR in save_results(): PermissionError\n"
            f"  Location: Cannot write to '{output_path}'\n"
            f"  Suggestion: Check that you have write permission for this path"
        )
        raise

    except OSError as e:
        # Catches disk full, bad path, and other OS-level write failures
        print(
            f"ERROR in save_results(): OSError\n"
            f"  Location: '{output_path}'\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check available disk space and that the folder exists"
        )
        raise


# ── Main Orchestrator ──────────────────────────────────────────────────────────
# Single responsibility: call the four modules above in the right order.
# No business logic lives here — if something needs to change, it changes
# in the relevant module, not here.

def main(json_path: str = "products.json", output_path: str = "results.json") -> None:
    """Run the full pipeline: load → validate → generate → save."""

    # API key from environment variable — never hardcoded
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print(
            f"ERROR in main(): Missing API key\n"
            f"  Suggestion: Set the OPENAI_API_KEY environment variable"
        )
        return

    # One client shared across all API calls
    api_client = OpenAI(api_key=api_key)

    # Each line below is one responsibility, one function
    products = load_and_validate_products(json_path)    # load + validate
    results  = process_products(products, api_client)   # generate descriptions
    save_results(results, output_path)                  # write to disk


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()

ℹ️  Loaded 3 valid product(s) from 'products.json'
⏳ Generating description for 'Wireless Bluetooth Headphones'...
✅ Done: 'Wireless Bluetooth Headphones'
⏳ Generating description for 'Smart Watch'...
✅ Done: 'Smart Watch'
⏳ Generating description for 'Laptop Stand'...
✅ Done: 'Laptop Stand'
ℹ️  Processed 3 of 3 product(s) successfully
✅ Results saved to 'results.json' (3 item(s))


In [8]:
# ── STEP 4: ERROR HANDLING ────────────────────────────────────────────────────
#
# Every function below follows the same format:
#
#   ERROR in function_name(): ErrorType
#     Location: where exactly it happened
#     Message:  what the error said
#     Suggestion: what to do about it
#
# No error ever fails silently. Every except block prints before raising.
# ─────────────────────────────────────────────────────────────────────────────

import json
import os
import httpx
from typing import Optional, List
from pydantic import BaseModel, field_validator, ValidationError
from openai import OpenAI, APIError, AuthenticationError, RateLimitError


# ── 1. load_json_file ──────────────────────────────────────────────────────────
# Handles: FileNotFoundError, JSONDecodeError
# Shows:   file path, current directory, line/column number for JSON errors

def load_json_file(file_path: str) -> dict:
    """Load and parse a JSON file. Raises on any failure after printing context."""
    try:
        with open(file_path, 'r') as f:
            return json.load(f)

    except FileNotFoundError:
        print(
            f"ERROR in load_json_file(): FileNotFoundError\n"
            f"  Location: File '{file_path}' not found\n"
            f"  Current directory: {os.getcwd()}\n"
            f"  Suggestion: Check the file path and confirm the file exists"
        )
        raise

    except json.JSONDecodeError as e:
        # e.lineno and e.colno pinpoint exactly where the syntax broke
        print(
            f"ERROR in load_json_file(): JSONDecodeError\n"
            f"  Location: File '{file_path}', line {e.lineno}, column {e.colno}\n"
            f"  Message: {e.msg}\n"
            f"  Suggestion: Fix the JSON syntax at line {e.lineno}"
        )
        raise

    except OSError as e:
        # Catches permission errors, locked files, unreadable paths
        print(
            f"ERROR in load_json_file(): OSError\n"
            f"  Location: File '{file_path}'\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check file permissions and that the path is accessible"
        )
        raise


# ── 2. validate_product_data ───────────────────────────────────────────────────
# Handles: ValidationError
# Shows:   product ID, every invalid field and why it failed
# Returns: None instead of raising so the caller can skip and continue

def validate_product_data(product_dict: dict) -> Optional[Product]:
    """Validate a raw dict against the Product model. Returns None on failure."""
    try:
        return Product(**product_dict)

    except ValidationError as e:
        # Build per-field breakdown so the user knows exactly what to fix
        field_errors = "\n".join(
            f"    - field '{'.'.join(str(loc) for loc in err['loc'])}': {err['msg']}"
            for err in e.errors()
        )
        print(
            f"ERROR in validate_product_data(): ValidationError\n"
            f"  Product ID: {product_dict.get('id', 'unknown')}\n"
            f"  Invalid fields:\n{field_errors}\n"
            f"  Suggestion: Fix the fields listed above in your JSON data"
        )
        return None   # caller decides to skip, not crash

    except Exception as e:
        # Fallback: unexpected error (e.g. product_dict is not a dict)
        print(
            f"ERROR in validate_product_data(): Unexpected {type(e).__name__}\n"
            f"  Location: Product dict: {product_dict}\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Confirm each item in 'products' is a JSON object"
        )
        return None


# ── 3. generate_description ────────────────────────────────────────────────────
# Handles: AuthenticationError, RateLimitError, httpx.TimeoutException,
#          httpx.ConnectError, generic APIError
# Shows:   product name + ID, error type, status code, message
# Raises:  always — process_products() decides whether to skip or abort

def generate_description(product: Product, api_client: OpenAI) -> str:
    """Call the OpenAI API for one product. Raises on any API failure."""
    prompt = create_product_prompt(product)

    try:
        response = api_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )
        return parse_api_response(response)

    except AuthenticationError as e:
        # Wrong or missing API key — no point retrying
        print(
            f"ERROR in generate_description(): AuthenticationError\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check that OPENAI_API_KEY is set correctly in your .env"
        )
        raise

    except RateLimitError as e:
        # Too many requests or quota exceeded
        print(
            f"ERROR in generate_description(): RateLimitError\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Wait a moment and retry, or check your usage quota"
        )
        raise

    except httpx.TimeoutException as e:
        # Network timeout — server took too long to respond
        print(
            f"ERROR in generate_description(): TimeoutException\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check your internet connection or increase timeout"
        )
        raise

    except httpx.ConnectError as e:
        # No internet / DNS failure / firewall blocking the request
        print(
            f"ERROR in generate_description(): ConnectError\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check your internet connection"
        )
        raise

    except APIError as e:
        # Catch-all for any other OpenAI API error
        print(
            f"ERROR in generate_description(): APIError\n"
            f"  Product: '{product.name}' (ID: {product.id})\n"
            f"  Error type: {type(e).__name__}\n"
            f"  Status code: {getattr(e, 'status_code', 'N/A')}\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check API key, rate limits, or try again later"
        )
        raise


# ── 4. save_results ────────────────────────────────────────────────────────────
# Handles: PermissionError, OSError
# Shows:   output path, OS error message

def save_results(results: List[dict], output_path: str) -> None:
    """Write results to a JSON file. Raises on any write failure."""
    try:
        with open(output_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"✅ Results saved to '{output_path}' ({len(results)} item(s))")

    except PermissionError:
        print(
            f"ERROR in save_results(): PermissionError\n"
            f"  Location: Cannot write to '{output_path}'\n"
            f"  Message: Write permission denied\n"
            f"  Suggestion: Check folder permissions or choose a different output path"
        )
        raise

    except OSError as e:
        # Covers disk full, invalid path characters, folder does not exist
        print(
            f"ERROR in save_results(): OSError\n"
            f"  Location: '{output_path}'\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check disk space and that the output folder exists"
        )
        raise

In [9]:
# ── TESTS: trigger every error type ───────────────────────────────────────────

# ── Test 1a: FileNotFoundError ─────────────────────────────────────────────────
print("=== Test 1a: missing file ===")
try:
    load_json_file("missing.json")
except FileNotFoundError:
    print("↑ correct — FileNotFoundError described above\n")

# ── Test 1b: JSONDecodeError ───────────────────────────────────────────────────
print("=== Test 1b: malformed JSON ===")
try:
    load_json_file("malformed.json")   # file created in Step 2 setup
except json.JSONDecodeError:
    print("↑ correct — JSONDecodeError with line/column described above\n")

# ── Test 2a: ValidationError — negative price ──────────────────────────────────
print("=== Test 2a: negative price ===")
result = validate_product_data({"id": "P099", "name": "Bad Item",
                                 "category": "Test", "price": -5.00})
print(f"↑ returned: {result}\n")   # should be None

# ── Test 2b: ValidationError — missing required field ─────────────────────────
print("=== Test 2b: missing name field ===")
result = validate_product_data({"id": "P100", "category": "Test", "price": 10.00})
print(f"↑ returned: {result}\n")   # should be None

# ── Test 3: load valid file ────────────────────────────────────────────────────
print("=== Test 3: valid file loads correctly ===")
data = load_json_file("products.json")
print(f"✅ Loaded {len(data['products'])} products\n")

# ── Test 4: save_results ───────────────────────────────────────────────────────
print("=== Test 4: save results ===")
save_results([{"product_id": "P001", "name": "Test", "description": "Works"}],
             "test_output.json")

=== Test 1a: missing file ===
ERROR in load_json_file(): FileNotFoundError
  Location: File 'missing.json' not found
  Current directory: c:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\Ironhack_Day7
  Suggestion: Check the file path and confirm the file exists
↑ correct — FileNotFoundError described above

=== Test 1b: malformed JSON ===
ERROR in load_json_file(): JSONDecodeError
  Location: File 'malformed.json', line 1, column 15
  Message: Expecting value
  Suggestion: Fix the JSON syntax at line 1
↑ correct — JSONDecodeError with line/column described above

=== Test 2a: negative price ===
ERROR in validate_product_data(): ValidationError
  Product ID: P099
  Invalid fields:
    - field 'price': Value error, Price must be positive
  Suggestion: Fix the fields listed above in your JSON data
↑ returned: None

=== Test 2b: missing name field ===
ERROR in validate_product_data(): ValidationError
  Product ID: P100
  Invalid fields:
    - field 'name': Field required
  Suggestion: Fix th

In [10]:
# ── STEP 5: FULL PIPELINE TESTS ───────────────────────────────────────────────
#
# Tests every scenario the lab requires:
#   5.1 — valid data, full pipeline (load → validate → generate → save)
#   5.2 — missing file
#   5.3 — malformed JSON
#   5.4 — invalid product data (bad fields)
#   5.5 — simulated API error (no real API call needed)
#
# Each test is independent — run them in any order.
# ─────────────────────────────────────────────────────────────────────────────

from dotenv import load_dotenv
from unittest.mock import MagicMock, patch
from openai import APIError
import os

load_dotenv()


# ── Test 5.1: Valid data — full pipeline ───────────────────────────────────────
# Runs the complete main() flow against products.json
# Expects: 3 descriptions generated and saved to results.json

print("=" * 60)
print("TEST 5.1 — Valid data, full pipeline")
print("=" * 60)

main(json_path="products.json", output_path="results.json")

# Confirm results.json was actually written and contains 3 items
with open("results.json") as f:
    saved = json.load(f)

print(f"\n✅ results.json contains {len(saved)} item(s)")
for item in saved:
    # Print first 80 chars of each description so output stays readable
    preview = item['description'][:80].replace('\n', ' ')
    print(f"   {item['product_id']} | {item['name']}")
    print(f"   → {preview}...")

TEST 5.1 — Valid data, full pipeline
ℹ️  Loaded 3 valid product(s) from 'products.json'
⏳ Generating description for 'Wireless Bluetooth Headphones'...
✅ Done: 'Wireless Bluetooth Headphones'
⏳ Generating description for 'Smart Watch'...
✅ Done: 'Smart Watch'
⏳ Generating description for 'Laptop Stand'...
✅ Done: 'Laptop Stand'
ℹ️  Processed 3 of 3 product(s) successfully
✅ Results saved to 'results.json' (3 item(s))

✅ results.json contains 3 item(s)
   P001 | Wireless Bluetooth Headphones
   → Elevate your listening experience with our Wireless Bluetooth Headphones, engine...
   P002 | Smart Watch
   → Elevate your lifestyle with our cutting-edge Smart Watch, a perfect blend of sty...
   P003 | Laptop Stand
   → Elevate your productivity and comfort with our premium Laptop Stand, designed to...


In [12]:
# ── Test 5.2: Missing file ─────────────────────────────────────────────────────
# Passes a filename that does not exist
# Expects: ERROR message showing the path and current directory

print("=" * 60)
print("TEST 5.2 — Missing file")
print("=" * 60)

try:
    load_and_validate_products("ghost_file.json")
except FileNotFoundError:
    print("↑ correct — pipeline stopped with clear FileNotFoundError")

TEST 5.2 — Missing file
ERROR in load_json_file(): FileNotFoundError
  Location: File 'ghost_file.json' not found
  Current directory: c:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\Ironhack_Day7
  Suggestion: Check the file path and confirm the file exists
↑ correct — pipeline stopped with clear FileNotFoundError


In [11]:
# ── Test 5.3: Malformed JSON ───────────────────────────────────────────────────
# malformed.json contains '{"products": [}' — invalid syntax
# Expects: ERROR message showing line number and column

print("=" * 60)
print("TEST 5.3 — Malformed JSON")
print("=" * 60)

try:
    load_and_validate_products("malformed.json")
except json.JSONDecodeError:
    print("↑ correct — pipeline stopped with line/column shown above")

TEST 5.3 — Malformed JSON
ERROR in load_json_file(): JSONDecodeError
  Location: File 'malformed.json', line 1, column 15
  Message: Expecting value
  Suggestion: Fix the JSON syntax at line 1
↑ correct — pipeline stopped with line/column shown above


# ── STEP 4 RESULTS SUMMARY ────────────────────────────────────────────────────
#
# All error handling verified. Results:
#
# Test 1a — FileNotFoundError
#              Shows: file path + current directory + suggestion
#
# Test 1b — JSONDecodeError
#              Shows: file name + line 1, column 15 + suggestion
#
# Test 2a — ValidationError (negative price)
#              Shows: product ID + field 'price' + reason + suggestion
#              Returns: None (does not crash the loop)
#
# Test 2b — ValidationError (missing required field)
#              Shows: product ID + missing field + suggestion
#              Returns: None (does not crash the loop)
#
# Test 3  — Valid file loads correctly
#              Shows: 3 products loaded with no errors
#
# Test 4  — save_results()
#              Shows: output path + item count confirmed
#
# Critical principle satisfied: NO SILENT FAILURES
# Every error prints function name, location, message, and suggestion
# before raising or returning None.
# ─────────────────────────────────────────────────────────────────────────────

In [13]:
# ── Test 5.4: Invalid product data ────────────────────────────────────────────
# invalid_products.json has a negative price and a missing name field
# Expects: ValidationError printed per bad product, valid ones still processed
# Important: pipeline should NOT crash — bad products are skipped, good ones continue

print("=" * 60)
print("TEST 5.4 — Invalid product data")
print("=" * 60)

# Load and validate only (no API calls) so we can inspect what survives
raw = load_json_file("invalid_products.json")

valid_products = []
for item in raw.get("products", []):
    product = validate_product_data(item)
    if product is not None:
        valid_products.append(product)

print(f"\n↑ {len(valid_products)} product(s) passed validation "
      f"(invalid ones skipped, not crashed)")

TEST 5.4 — Invalid product data
ERROR in validate_product_data(): ValidationError
  Product ID: P004
  Invalid fields:
    - field 'price': Value error, Price must be positive
  Suggestion: Fix the fields listed above in your JSON data
ERROR in validate_product_data(): ValidationError
  Product ID: P005
  Invalid fields:
    - field 'name': Field required
  Suggestion: Fix the fields listed above in your JSON data

↑ 0 product(s) passed validation (invalid ones skipped, not crashed)


In [14]:
# ── Test 5.5: Simulated API error ─────────────────────────────────────────────
# Uses unittest.mock to fake an APIError without making a real API call
# Expects: ERROR message showing product name, error type, status code

print("=" * 60)
print("TEST 5.5 — API error (simulated)")
print("=" * 60)

# Build a real Product object to pass in
test_product = Product(
    id="P001",
    name="Wireless Headphones",
    category="Electronics",
    price=99.99,
    features=["Bluetooth", "Noise Cancelling"]
)

# Create a mock client whose .chat.completions.create() raises APIError
mock_client = MagicMock()
mock_client.chat.completions.create.side_effect = APIError(
    message="Service temporarily unavailable",
    request=MagicMock(),
    body={}
)

try:
    generate_description(test_product, mock_client)
except APIError:
    print("↑ correct — APIError caught with product context shown above")

TEST 5.5 — API error (simulated)
ERROR in generate_description(): APIError
  Product: 'Wireless Headphones' (ID: P001)
  Error type: APIError
  Status code: N/A
  Message: Service temporarily unavailable
  Suggestion: Check API key, rate limits, or try again later
↑ correct — APIError caught with product context shown above


In [15]:
# ── STEP 5 SUMMARY ────────────────────────────────────────────────────────────

print("""
STEP 5 RESULTS
══════════════════════════════════════════════════════════════
✅ 5.1  Valid pipeline     — 3 products loaded, described, saved
✅ 5.2  Missing file       — FileNotFoundError with path + directory
✅ 5.3  Malformed JSON     — JSONDecodeError with line + column
✅ 5.4  Invalid data       — ValidationError per field, pipeline continues
✅ 5.5  API error          — APIError with product name + type + suggestion

Critical principle confirmed: NO SILENT FAILURES
Every error tells you WHAT went wrong, WHERE, and WHAT TO DO.
══════════════════════════════════════════════════════════════
""")


STEP 5 RESULTS
══════════════════════════════════════════════════════════════
✅ 5.1  Valid pipeline     — 3 products loaded, described, saved
✅ 5.2  Missing file       — FileNotFoundError with path + directory
✅ 5.3  Malformed JSON     — JSONDecodeError with line + column
✅ 5.4  Invalid data       — ValidationError per field, pipeline continues
✅ 5.5  API error          — APIError with product name + type + suggestion

Critical principle confirmed: NO SILENT FAILURES
Every error tells you WHAT went wrong, WHERE, and WHAT TO DO.
══════════════════════════════════════════════════════════════



# ── SILENT FAILURES RESOLVED ──────────────────────────────────────────────────
#
# Original code had 4 silent failures. All fixed:
#
# 1. bare except: pass in validation loop
#    → replaced with ValidationError that prints which fields failed
#      and returns None so the loop continues instead of crashing
#
# 2. open(json_file) with no error handling
#    → now catches FileNotFoundError (shows path + cwd) and
#      JSONDecodeError (shows line + column number)
#
# 3. API call with no try/except
#    → now catches AuthenticationError, RateLimitError, TimeoutException,
#      ConnectError, and generic APIError — each prints the product name
#      and a specific suggestion before raising
#
# 4. results only saved at the very end
#    → process_products() catches APIError per product and skips it
#      so partial results are never lost if one API call fails
# ─────────────────────────────────────────────────────────────────────────────

In [16]:
# ── STEP 6: OpenAI API WRAPPER ────────────────────────────────────────────────
#
# Wraps the OpenAI client into a reusable class so that:
#   - retry logic lives in one place and is never duplicated
#   - exponential backoff is automatic (1s, 2s, 4s between attempts)
#   - timeout is configurable at instantiation, not scattered in each call
#   - every failure prints WHERE it occurred and which attempt it was
#   - the rest of the code just calls wrapper.generate_description()
#     and never thinks about retries or error handling again
# ─────────────────────────────────────────────────────────────────────────────

import time
import os
from openai import OpenAI, APIError, AuthenticationError, RateLimitError
import httpx


class OpenAIWrapper:
    """
    Reusable wrapper for OpenAI API calls.
    Handles retry logic, exponential backoff, timeout, and error reporting.
    """

    def __init__(self, api_key: str, max_retries: int = 3, timeout: int = 30):
        # timeout applies to every request made through this client
        self.client = OpenAI(api_key=api_key, timeout=timeout)
        self.max_retries = max_retries
        self.timeout = timeout
        print(f"✅ OpenAIWrapper ready "
              f"(max_retries={max_retries}, timeout={timeout}s)")

    def generate_description(self, prompt: str, model: str = "gpt-4o-mini") -> str:
        """
        Call the API with automatic retry and exponential backoff.
        Retries on RateLimitError and generic APIError.
        Does NOT retry on AuthenticationError — wrong key won't fix itself.
        Raises on final failed attempt after printing full context.
        """

        for attempt in range(self.max_retries):

            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}]
                )
                # Success — return immediately, no retries needed
                return parse_api_response(response)

            except AuthenticationError as e:
                # Wrong API key — retrying will never help, fail immediately
                print(
                    f"ERROR in OpenAIWrapper.generate_description(): "
                    f"AuthenticationError\n"
                    f"  Attempt: {attempt + 1} of {self.max_retries}\n"
                    f"  Message: {str(e)}\n"
                    f"  Suggestion: Check OPENAI_API_KEY in your .env file — "
                    f"no retry attempted"
                )
                raise   # no backoff loop, exit immediately

            except RateLimitError as e:
                # Quota hit — worth retrying after a wait
                wait_time = 2 ** attempt     # 1s → 2s → 4s
                if attempt < self.max_retries - 1:
                    print(
                        f"WARNING in OpenAIWrapper.generate_description(): "
                        f"RateLimitError\n"
                        f"  Attempt: {attempt + 1} of {self.max_retries}\n"
                        f"  Message: {str(e)}\n"
                        f"  Retrying in {wait_time}s..."
                    )
                    time.sleep(wait_time)
                else:
                    # Final attempt also failed
                    print(
                        f"ERROR in OpenAIWrapper.generate_description(): "
                        f"RateLimitError — all {self.max_retries} attempts failed\n"
                        f"  Message: {str(e)}\n"
                        f"  Suggestion: Wait before retrying or check usage quota"
                    )
                    raise

            except httpx.TimeoutException as e:
                # Server took longer than self.timeout seconds
                wait_time = 2 ** attempt
                if attempt < self.max_retries - 1:
                    print(
                        f"WARNING in OpenAIWrapper.generate_description(): "
                        f"TimeoutException\n"
                        f"  Attempt: {attempt + 1} of {self.max_retries} "
                        f"(timeout={self.timeout}s)\n"
                        f"  Retrying in {wait_time}s..."
                    )
                    time.sleep(wait_time)
                else:
                    print(
                        f"ERROR in OpenAIWrapper.generate_description(): "
                        f"TimeoutException — all {self.max_retries} attempts failed\n"
                        f"  Message: {str(e)}\n"
                        f"  Suggestion: Check connection or increase timeout"
                    )
                    raise

            except APIError as e:
                # Catch-all for any other OpenAI error — retry with backoff
                wait_time = 2 ** attempt
                if attempt < self.max_retries - 1:
                    print(
                        f"WARNING in OpenAIWrapper.generate_description(): "
                        f"{type(e).__name__}\n"
                        f"  Attempt: {attempt + 1} of {self.max_retries}\n"
                        f"  Status code: {getattr(e, 'status_code', 'N/A')}\n"
                        f"  Message: {str(e)}\n"
                        f"  Retrying in {wait_time}s..."
                    )
                    time.sleep(wait_time)
                else:
                    print(
                        f"ERROR in OpenAIWrapper.generate_description(): "
                        f"{type(e).__name__} — all {self.max_retries} attempts failed\n"
                        f"  Status code: {getattr(e, 'status_code', 'N/A')}\n"
                        f"  Message: {str(e)}\n"
                        f"  Suggestion: Check API key, rate limits, or try later"
                    )
                    raise

In [17]:
# ── STEP 7: LOGGING ───────────────────────────────────────────────────────────
#
# Adds a complete audit trail to every operation in the pipeline.
# Two outputs simultaneously:
#   - timestamped .log file on disk (permanent record)
#   - stdout stream (visible in notebook output)
#
# Four log levels used:
#   DEBUG   — raw API responses (verbose, useful when debugging)
#   INFO    — normal operations (file loaded, product validated, etc.)
#   WARNING — recoverable problems (retry attempt, product skipped)
#   ERROR   — failures that stop a product or the pipeline
# ─────────────────────────────────────────────────────────────────────────────

import logging
from datetime import datetime

# Timestamped filename so each run creates a fresh log — old logs are never overwritten
log_filename = f"product_generator_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.DEBUG,         # capture everything DEBUG and above
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filename),   # full log written to disk
        logging.StreamHandler()              # same output visible in notebook
    ]
)

logger = logging.getLogger(__name__)
logger.info(f"Logging initialised — writing to '{log_filename}'")

2026-04-30 12:15:52,199 - __main__ - INFO - Logging initialised — writing to 'product_generator_20260430_121552.log'


In [18]:
# ── STEP 7 TEST — run pipeline and inspect log file ───────────────────────────

main(json_path="products.json", output_path="results_logged.json")

# Print the log file so it is visible inside the notebook
print(f"\n{'=' * 60}")
print(f"LOG FILE: {log_filename}")
print('=' * 60)
with open(log_filename) as f:
    print(f.read())

2026-04-30 12:16:31,203 - openai._base_client - DEBUG - Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-e341bb6d-838c-442c-9230-a25ef604798a', 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Create a compelling product description for an online store listing.\n\nProduct name: Wireless Bluetooth Headphones\nCategory: Electronics\nPrice: $99.99\nKey features: Bluetooth 5.0, Noise Cancelling, 30hr Battery\n\nWrite 2-3 sentences. Focus on benefits, not just features. Tone: enthusiastic but professional.'}], 'model': 'gpt-4o-mini'}}
2026-04-30 12:16:31,206 - openai._base_client - DEBUG - Sending HTTP Request: POST https://api.openai.com/v1/chat/completions
2026-04-30 12:16:31,207 - httpcore.connection - DEBUG - connect_tcp.started host='api.openai.com' port=443 local_address=None timeout=5.0 socket_options=None
2026-04-30 12:16:31,246 - httpcore.connection - DEBUG - connect_tcp.complete return

ℹ️  Loaded 3 valid product(s) from 'products.json'
⏳ Generating description for 'Wireless Bluetooth Headphones'...


2026-04-30 12:16:34,179 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Apr 2026 10:16:33 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9f45ca98fca5b84a-FRA'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'ironhack-engineering'), (b'openai-processing-ms', b'1918'), (b'openai-project', b'proj_BcfRS3AwrK6N0lWaWWGKaWhX'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'30000'), (b'x-ratelimit-limit-tokens', b'150000000'), (b'x-ratelimit-remaining-requests', b'29999'), (b'x-ratelimit-remaining-tokens', b'149999922'), (b'x-ratelimit-reset-requests', b'2ms'), (b'x-rat

✅ Done: 'Wireless Bluetooth Headphones'
⏳ Generating description for 'Smart Watch'...


2026-04-30 12:16:36,953 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Apr 2026 10:16:36 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9f45caab3cf5b84a-FRA'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'ironhack-engineering'), (b'openai-processing-ms', b'1594'), (b'openai-project', b'proj_BcfRS3AwrK6N0lWaWWGKaWhX'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'30000'), (b'x-ratelimit-limit-tokens', b'150000000'), (b'x-ratelimit-remaining-requests', b'29999'), (b'x-ratelimit-remaining-tokens', b'149999927'), (b'x-ratelimit-reset-requests', b'2ms'), (b'x-rat

✅ Done: 'Smart Watch'
⏳ Generating description for 'Laptop Stand'...


2026-04-30 12:16:39,403 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Apr 2026 10:16:38 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9f45cabc8a6db84a-FRA'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'ironhack-engineering'), (b'openai-processing-ms', b'1965'), (b'openai-project', b'proj_BcfRS3AwrK6N0lWaWWGKaWhX'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'30000'), (b'x-ratelimit-limit-tokens', b'150000000'), (b'x-ratelimit-remaining-requests', b'29999'), (b'x-ratelimit-remaining-tokens', b'149999925'), (b'x-ratelimit-reset-requests', b'2ms'), (b'x-rat

✅ Done: 'Laptop Stand'
ℹ️  Processed 3 of 3 product(s) successfully
✅ Results saved to 'results_logged.json' (3 item(s))

LOG FILE: product_generator_20260430_121552.log
2026-04-30 12:15:52,199 - __main__ - INFO - Logging initialised — writing to 'product_generator_20260430_121552.log'
2026-04-30 12:16:31,203 - openai._base_client - DEBUG - Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-e341bb6d-838c-442c-9230-a25ef604798a', 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Create a compelling product description for an online store listing.\n\nProduct name: Wireless Bluetooth Headphones\nCategory: Electronics\nPrice: $99.99\nKey features: Bluetooth 5.0, Noise Cancelling, 30hr Battery\n\nWrite 2-3 sentences. Focus on benefits, not just features. Tone: enthusiastic but professional.'}], 'model': 'gpt-4o-mini'}}
2026-04-30 12:16:31,206 - openai._base_client - DEBUG - Sending HTTP Request: PO